In [ ]:
import sys
sys.path.insert(0, "..")

from typing import Literal
from pydantic import BaseModel, Field


class Fact(BaseModel):
    measure: Literal["HR", "median", "rate", "difference", "n", "events", "p", "other"] = Field(
        description="что это за число: HR; median — медиана; rate — доля или частота, %; difference — разница; "
                    "n — число пациентов; events — число событий; p — p-значение; other — другое")
    endpoint: str = Field(description="конечная точка коротко, как на слайде: PFS, OS, DFS, ORR, CR, DOR, pCR; "
                                      "для характеристик пациентов — baseline")
    group: str = Field(description="к какой группе, сравнению или подгруппе относится: 'EV+P', 'EV+P vs chemotherapy', "
                                   "'all patients', 'age ≥65'")
    value: str = Field(description="значение ровно как напечатано, без единиц: '0.58', '<0.001', '33.6', 'NR'")
    unit: str | None = Field(description="единица, если напечатана: '%', 'mo'; иначе null")
    ci_level: str | None = Field(description="уровень доверительного интервала, как напечатан: '95', '99.5'; иначе null")
    ci_low: str | None = Field(description="нижняя граница интервала, как напечатана; иначе null")
    ci_high: str | None = Field(description="верхняя граница интервала, как напечатана; иначе null")
    timepoint: str | None = Field(description="срок, если указан: '3 yr', '36 mo', '4-year'; иначе null")


class SlideFacts(BaseModel):
    title: str | None = Field(description="заголовок слайда, как напечатан")
    trial_name: str | None = Field(description="название исследования, как напечатано на слайде")
    nct: str | None = Field(description="номер NCT, если напечатан; иначе null")
    citation: str | None = Field(description="ссылка на публикацию или конференцию, как напечатана; иначе null")
    slide_type: Literal["results", "design", "baseline", "text", "other"] = Field(
        description="results — результаты; design — дизайн исследования; baseline — характеристики пациентов; "
                    "text — текст без таблиц и графиков; other — другое")
    facts: list[Fact] = Field(description="все числовые результаты со слайда; у дизайна исследования — пустой список")
    uncertain: list[str] = Field(description="что прочитано неуверенно или не читается — каждое отдельной строкой")


example = Fact(measure="HR", endpoint="IDFS", group="olaparib vs placebo", value="0.58", unit=None,
               ci_level="99.5", ci_low="0.41", ci_high="0.82", timepoint=None)
print(example)
print("бланки внутри:", list(SlideFacts.model_json_schema()["$defs"]))
print("граф у факта:", len(Fact.model_fields))

In [ ]:
import base64
import io
import json
import time
from datetime import datetime
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from PIL import Image

from app.config import cost_usd, READ_MODEL

load_dotenv("../.env")
client = anthropic.Anthropic()
RUN_PAID = False   # предохранитель: True — только когда сознательно запускаешь платный прогон


def image_to_base64(img):
    """Картинку Pillow → строка base64 в формате PNG (без записи на диск)."""
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    return base64.standard_b64encode(buffer.getvalue()).decode("utf-8")


def read_facts(images, model, instruction):
    """Картинки слайда → бланк SlideFacts от модели; вернуть бланк, расход, время и причину остановки."""
    if not RUN_PAID:
        raise RuntimeError("платный вызов выключен: поставь RUN_PAID = True")
    content = []
    for img in images:
        content.append({"type": "image",
                        "source": {"type": "base64", "media_type": "image/png", "data": image_to_base64(img)}})
    content.append({"type": "text", "text": instruction})
    t0 = time.perf_counter()
    response = client.messages.parse(
        model=model,
        max_tokens=16000,
        messages=[{"role": "user", "content": content}],
        output_format=SlideFacts,
    )
    return {
        "model": model,
        "reading": response.parsed_output.model_dump() if response.parsed_output else None,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
        "cost_usd": cost_usd(response.usage, model),
        "seconds": round(time.perf_counter() - t0, 1),
        "stop_reason": response.stop_reason,
    }


def save_run(run, name):
    """Сохранить прогон в data/runs/<дата>_<имя>.json; вернуть путь."""
    folder = Path("../data/runs")
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_{name}.json"
    path.write_text(json.dumps(run, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


print("модель чтения:", READ_MODEL)

In [ ]:
INSTRUCTION_DRAFT = (
    "Это слайд доклада. Перепиши в бланк ВСЕ числовые результаты, которые напечатаны на слайде, "
    "ровно как напечатаны: каждое число — отдельным фактом, интервал — в графы ci_* того числа, к которому он относится. "
    "Ничего не додумывай и не бери из памяти: только то, что видно на слайде. "
    "Числа под кривыми «No. at risk» и оси графиков не выписывай. "
    "Если число не читается — не угадывай: пропусти его и добавь пункт в uncertain."
)

RUN_PAID = True                                              # сознательно: один прогон Opus, ≈ $0,10–0,20
olympia = Image.open(sorted(Path("../tests/slides/real").glob("*.png"))[0]).convert("RGB")
draft = read_facts([olympia], READ_MODEL, INSTRUCTION_DRAFT)
RUN_PAID = False                                             # сразу выключить обратно
print(save_run(draft, "draft_olympia"))

print(f"${draft['cost_usd']:.4f} · {draft['seconds']} с · {draft['stop_reason']} · выход {draft['output_tokens']} токенов")
reading = draft["reading"]
print("исследование:", reading["trial_name"], "· тип:", reading["slide_type"], "· фактов:", len(reading["facts"]))
for f in reading["facts"]:
    ci = f"({f['ci_level']}% ДИ {f['ci_low']}–{f['ci_high']})" if f["ci_low"] else ""
    print(f"  {f['measure']:10} {f['endpoint']:8} {f['group'][:34]:34} {f['value']:>8} {f['unit'] or '':3} {ci}")
print("сомнения:", reading["uncertain"])

In [ ]:
gold = json.loads(Path("../tests/gold/olympia_2021.json").read_text(encoding="utf-8"))
MEASURE_OF = {"rate": "rate", "diff": "difference", "hr": "HR", "p": "p", "events": "events"}   # поле эталона → показатель бланка v1


def same_printed(printed, expected):
    """Совпадает ли напечатанное значение с эталоном: числа — как числа (13.0 = 13), строки — без пробелов."""
    if isinstance(expected, str):
        return printed.replace(" ", "") == expected.replace(" ", "")
    try:
        return float(printed) == float(expected)
    except ValueError:
        return False


def found_in_draft(facts, code, field, expected):
    """Есть ли в черновике факт с той же конечной точкой, показателем, группой и значением, что поле эталона."""
    kind = field.split("_")[0]                                   # rate / diff / hr / p / events
    arm = field.split("_")[-1] if kind in ("rate", "events") else None   # olaparib / placebo
    for fact in facts:
        if fact["endpoint"].upper() != code or fact["measure"] != MEASURE_OF[kind]:
            continue
        if arm and arm not in fact["group"].lower():
            continue
        slot = fact[field[field.index("ci_"):]] if "ci_" in field else fact["value"]   # ci_low / ci_high / ci_level или само число
        if slot is not None and same_printed(slot, expected):
            return True
    return False


facts = draft["reading"]["facts"]
missing = []
for code, fields in gold["endpoints"].items():
    for field, expected in fields.items():
        if not found_in_draft(facts, code, field, expected):
            missing.append(f"{code}.{field} = {expected}")
print("черновик Opus: найдено", 22 - len(missing), "из 22 чисел эталона на своём месте · нет:", missing)
print("контроль (HR IDFS 0.85 — такого нет):", found_in_draft(facts, "IDFS", "hr", 0.85))

In [ ]:
import cv2
from app.capture import capture

for name in ["IMG-20230604-WA0010.jpg", "5249372745471040807.jpg", "IMG_20260410_180549.jpg"]:
    slide, outcome = capture(cv2.imread(f"../tests/slides/real/{name}"))
    print(name, "·", outcome, "·", None if slide is None else f"{slide.shape[1]} × {slide.shape[0]}")

In [ ]:
from app.ocr import run_ocr, hint_text

INSTRUCTION_HINT = (
    INSTRUCTION_DRAFT + "\n\n"
    "Ниже — текст этого слайда, распознанный программой OCR, с координатами центра каждой строки "
    "в процентах: x — слева направо, y — сверху вниз. Цифры в OCR точнее, чем на картинке: значения чисел "
    "бери из OCR, а по картинке и координатам определяй, к какой панели, кривой, группе и показателю "
    "относится число. Числа нет в OCR — читай с картинки и добавь пункт в uncertain.\n\n"
)

olympia_ocr = run_ocr(olympia)
print("OCR: строк", len(olympia_ocr["lines"]), "· чисел", len(olympia_ocr["numbers"]))

RUN_PAID = True                                              # сознательно: один прогон Haiku, ≈ $0,03–0,05
draft_haiku = read_facts([olympia], "claude-haiku-4-5", INSTRUCTION_HINT + hint_text(olympia_ocr))
RUN_PAID = False
print(save_run(draft_haiku, "draft_olympia_haiku_hint"))
print(f"${draft_haiku['cost_usd']:.4f} · {draft_haiku['seconds']} с · {draft_haiku['stop_reason']} · фактов: {len(draft_haiku['reading']['facts'])}")

missing = []
for code, fields in gold["endpoints"].items():
    for field, expected in fields.items():
        if not found_in_draft(draft_haiku["reading"]["facts"], code, field, expected):
            missing.append(f"{code}.{field} = {expected}")
print("Haiku + OCR: найдено", 22 - len(missing), "из 22 на своём месте · нет:", missing)

In [ ]:
REAL = Path("../tests/slides/real")
SLIDE_SET = {                                  # ключ — короткое имя слайда в наборе; значение — файл фото
    "01_dostarlimab_pfs":      "-5249372745471040796_121.jpg",
    "02_ev302_os":             "5249372745471040807.jpg",
    "03_ras_g12_pfs":          "5253871367231314989.jpg",
    "04_camizestrant_pfs":     "5253871367231314986.jpg",
    "05_bezuclastinib_orr":    "-5249372745471040801_121.jpg",
    "06_atezolizumab_table":   "5278289995770829329.jpg",
    "07_alkove1_baseline":     "-5247120945657356337_121.jpg",
    "08_elisrasib_waterfall":  "-5249372745471040805_121.jpg",
    "09_ev302_orr":            "5249372745471040808.jpg",
    "10_orr_subgroups":        "5254005559189511972.jpg",
    "11_drfi_hr":              "5249372745471040625.jpg",
    "12_oligoprogression":     "5249372745471040750.jpg",
    "13_keynote564_design":    "5249372745471040813.jpg",
    "14_adaura_os_stage":      "IMG-20230604-WA0010.jpg",
    "15_olympia":              sorted(REAL.glob("*.png"))[0].name,   # в имени невидимые пробелы macOS — берём поиском
}

slides = {}
for key, name in SLIDE_SET.items():
    photo = cv2.imread(str(REAL / name))
    if photo is None:                                          # imread не падает, а молча отдаёт None
        print(f"{key:24} ФАЙЛ НЕ ПРОЧИТАН — {name}")
        continue
    slide, outcome = capture(photo)
    if slide is None:
        print(f"{key:24} СЛАЙД НЕ НАЙДЕН — {name}")
        continue
    slides[key] = Image.fromarray(cv2.cvtColor(slide, cv2.COLOR_BGR2RGB))   # BGR OpenCV → RGB Pillow
    print(f"{key:24} {outcome:10} {slides[key].width} × {slides[key].height}")
print("слайдов готово:", len(slides), "из", len(SLIDE_SET))

sheet = Image.new("RGB", (4 * 320, 4 * 190), "white")         # лист-миниатюры: видно, что уходит в модели
for i, img in enumerate(slides.values()):
    thumb = img.copy()
    thumb.thumbnail((310, 180))
    sheet.paste(thumb, ((i % 4) * 320, (i // 4) * 190))
sheet

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

DRAFT_MODELS = {"opus": "claude-opus-5", "sonnet": "claude-sonnet-5"}   # два независимых черновика (решение № 34)
schema = anthropic.transform_schema(SlideFacts)                        # бланк → JSON-схема в том виде, который ждёт API

batch_requests = []
for key, img in slides.items():
    for short, model in DRAFT_MODELS.items():
        batch_requests.append(Request(
            custom_id=f"{key}__{short}",                               # по нему найдём ответ: результаты приходят в любом порядке
            params=MessageCreateParamsNonStreaming(
                model=model,
                max_tokens=16000,
                messages=[{"role": "user", "content": [
                    {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_to_base64(img)}},
                    {"type": "text", "text": INSTRUCTION_DRAFT},
                ]}],
                output_config={"format": {"type": "json_schema", "schema": schema}},
            ),
        ))
print("запросов в пакете:", len(batch_requests))

RUN_PAID = True                                                # сознательно: пакет черновиков, ≈ $1,0–1,5
batch = client.messages.batches.create(requests=batch_requests)
RUN_PAID = False
Path("../data/runs/batch_drafts_id.txt").write_text(batch.id)  # номер пакета — на диск: без него результаты не забрать
print("пакет:", batch.id, "·", batch.processing_status)

In [ ]:
batch_id = Path("../data/runs/batch_drafts_id.txt").read_text()
batch = client.messages.batches.retrieve(batch_id)
counts = batch.request_counts
print(batch.processing_status, "· готово:", counts.succeeded, "· ошибок:", counts.errored,
      "· в работе:", counts.processing, "· истекло:", counts.expired)

In [ ]:
from IPython.display import HTML, display
from html import escape
from collections import Counter
import re


def norm(text):
    """Строку → вид для сравнения: заглавные, без пробелов и дефисов; None → пустая строка."""
    return (text or "").upper().replace(" ", "").replace("-", "")


def same_place(fa, fb):
    """Одинаково ли два черновика поняли, что это за число: показатель и конечная точка."""
    return fa["measure"] == fb["measure"] and norm(fa["endpoint"]) == norm(fb["endpoint"])


def same_group(fa, fb):
    """Похожи ли группы: одна запись содержит другую («Olaparib» и «Olaparib, all patients»)."""
    a, b = norm(fa["group"]), norm(fb["group"])
    return a in b or b in a


def compare_drafts(facts_a, facts_b):
    """Два черновика → строки сверки: совпало / группа разная / место разное / только A / только B."""
    rows = []
    used = set()                                       # факты B, уже нашедшие пару
    for fa in facts_a:
        match = None
        for j, fb in enumerate(facts_b):
            if j in used or norm(fb["value"]) != norm(fa["value"]) or norm(fb["ci_low"]) != norm(fa["ci_low"]):
                continue
            if match is None or same_place(fa, fb):    # из одинаковых чисел предпочитаем то, что на том же месте
                match = j
            if same_place(fa, fb):
                break
        if match is None:
            rows.append(("только A", fa, None))
            continue
        used.add(match)
        fb = facts_b[match]
        if not same_place(fa, fb):
            status = "место разное"
        elif not same_group(fa, fb):
            status = "группа разная"
        else:
            status = "совпало"
        rows.append((status, fa, fb))
    for j, fb in enumerate(facts_b):
        if j not in used:
            rows.append(("только B", None, fb))
    return rows


def seen_by_ocr(value, ocr):
    """Видит ли OCR это значение на слайде: все числа из него — среди чисел OCR; без чисел — поиск в тексте."""
    numbers = re.findall(r"\d+(?:[.,]\d+)?", value)
    if not numbers:
        return norm(value).lower() in ocr["text"]
    return all(float(n.replace(",", ".")) in ocr["numbers"] for n in numbers)


COLORS = {"совпало": "#e6f4ea", "группа разная": "#fff4cc", "место разное": "#ffe0b2",
          "только A": "#fde2e1", "только B": "#fde2e1"}


def fact_text(f):
    """Факт → короткая строка для таблицы сверки; escape — чтобы «<0.001» не приняли за разметку."""
    if f is None:
        return "—"
    ci = f" ({f['ci_level']}% ДИ {f['ci_low']}–{f['ci_high']})" if f["ci_low"] else ""
    when = f" · {f['timepoint']}" if f["timepoint"] else ""
    return escape(f"{f['measure']} · {f['endpoint']} · {f['group']} · ") + f"<b>{escape(f['value'])}</b>" + escape(f"{ci}{when}")


def show_check(img, rows, ocr, name_a, name_b):
    """Слайд + таблица сверки: цвет — согласие черновиков, «OCR» — видит ли программа число на слайде."""
    display(img)
    html = [f"<table style='color:#111'><tr><th>№</th><th>статус</th><th>OCR</th><th>{name_a}</th><th>{name_b}</th></tr>"]
    for i, (status, fa, fb) in enumerate(rows):
        seen = "да" if seen_by_ocr((fa or fb)["value"], ocr) else "<b>НЕТ</b>"
        html.append(f"<tr style='background:{COLORS[status]}'><td>{i}</td><td>{status}</td><td>{seen}</td>"
                    f"<td>{fact_text(fa)}</td><td>{fact_text(fb)}</td></tr>")
    html.append("</table>")
    display(HTML("".join(html)))
    print(dict(Counter(status for status, fa, fb in rows)))


olympia_rows = compare_drafts(draft["reading"]["facts"], draft_haiku["reading"]["facts"])
show_check(olympia, olympia_rows, olympia_ocr, "Opus", "Haiku + OCR")

In [ ]:
from html import unescape
from datetime import date

GOLD_DIR = Path("../data/gold_set")          # эталон набора — не в git: расшифровка чужих слайдов (правило 36а)


def pick(rows, number, side="A", fix=None):
    """Строка таблицы сверки → ключевое утверждение: факт черновика A или B, при необходимости с правкой полей."""
    status, fa, fb = rows[number]
    chosen = fa if side == "A" else fb
    if chosen is None:
        raise ValueError(f"в строке {number} нет черновика {side} — возьми другую сторону")
    claim = dict(chosen)                     # копия: черновик не трогаем
    if fix:
        claim.update(fix)                    # например {"value": "79.6", "source": "Geyer 2022, Fig. 2C"} — решение № 35
    return claim


def save_gold(key, study, key_claims, slide_problems, questions):
    """Эталон слайда (решение № 36) → data/gold_set/<ключ>.json; вернуть путь."""
    if not 1 <= len(key_claims) <= 6 and study.get("type") != "design":
        print(f"внимание: ключевых утверждений {len(key_claims)} — по решению № 36 их 3–6")
    gold = {
        "slide": key,
        "study": study,
        "key_claims": key_claims,
        "slide_problems": slide_problems,
        "questions": questions,
        "verified_by": "Иван",
        "date": date.today().isoformat(),
    }
    GOLD_DIR.mkdir(parents=True, exist_ok=True)
    path = GOLD_DIR / f"{key}.json"
    path.write_text(json.dumps(gold, ensure_ascii=False, indent=2), encoding="utf-8")
    return path


def show_gold(path):
    """Показать сохранённый эталон коротко — проверить глазами, что записалось."""
    gold = json.loads(Path(path).read_text(encoding="utf-8"))
    print("исследование:", gold["study"])
    for claim in gold["key_claims"]:
        text = unescape(fact_text(claim).replace("<b>", "").replace("</b>", ""))
        print("  •", text, f"[{claim['source']}]" if claim.get("source") else "")
    print("проблемы слайда:", gold["slide_problems"])
    print("вопросы:", gold["questions"])

In [ ]:
path = save_gold(
    "15_olympia",
    study={"name": "OlympiA",
           "nct": "NCT02032823",
           "source": "Tutt NEJM 2021 (панели A, B); диаграмма подгрупп — Geyer Ann Oncol 2022, Fig. 2C"},
    key_claims=[
        pick(olympia_rows, 9),     # HR IDFS 0.58 (99.5% ДИ 0.41–0.82)
        pick(olympia_rows, 8),     # разница 3-летней IDFS 8.8 п.п. (95% ДИ 4.5–13.0)
        pick(olympia_rows, 20),    # HR DDFS 0.57 (99.5% ДИ 0.39–0.83)
        pick(olympia_rows, 24),    # HR DDFS 0.607 (95% ДИ 0.476–0.771) — 4-летний анализ, диаграмма подгрупп
    ],
    slide_problems=[
        "Название напечатано как «Olympiad», исследование — OlympiA",
        "Диаграмма подгрупп взята из Geyer 2022 (4-летний анализ), а подписана как Tutt 2021",
    ],
    questions=[
        "насколько релевантно сравнение с плацебо, отражает ли это реальную клиническую практику?",
        "какую гипотезы вы считаете однозначно доказанной по этому результату?"
        "изменит ли, на ваш взгляд, увеличение длительности наблюдения эти результаты - если да, то повлияет ли это на клиническую практику",
    ],
)
show_gold(path)

In [ ]:
QUICK = ["04_camizestrant_pfs", "02_ev302_os", "05_bezuclastinib_orr"]   # 3 слайда обычными вызовами, пока идёт пакет
quick_drafts = {}
quick_cost = 0.0

RUN_PAID = True                                     # сознательно: 3 слайда × Opus и Sonnet, ≈ $0,4
for key in QUICK:
    for short, model in DRAFT_MODELS.items():
        try:
            run = read_facts([slides[key]], model, INSTRUCTION_DRAFT)
        except Exception as error:
            print(f"{key:22} {short:6} ошибка: {type(error).__name__}")
            continue
        quick_cost += run["cost_usd"]
        save_run(run, f"draft_{key}_{short}")
        quick_drafts.setdefault(key, {})[short] = run["reading"]
        print(f"{key:22} {short:6} ${run['cost_usd']:.4f} · {run['seconds']} с · {run['stop_reason']} · фактов: {len(run['reading']['facts'])}")
RUN_PAID = False
print(f"расход: ${quick_cost:.4f}")

In [ ]:
key = "02_ev302_os"
rows = compare_drafts(quick_drafts[key]["opus"]["facts"], quick_drafts[key]["sonnet"]["facts"])
show_check(slides[key], rows, run_ocr(slides[key]), "Opus", "Sonnet")
print("исследование — Opus:", quick_drafts[key]["opus"]["trial_name"], "· Sonnet:", quick_drafts[key]["sonnet"]["trial_name"])
print("сомнения Opus:", quick_drafts[key]["opus"]["uncertain"])
print("сомнения Sonnet:", quick_drafts[key]["sonnet"]["uncertain"])

In [ ]:
path = save_gold(
    "05_bezuclastinib_orr",
    study={"name": "SERENA-6", "nct": "NCT04964934",
           "source": "Bidard FC et al., NEJM 2025;393:569–580 (на слайде название не напечатано)"},
    key_claims=[
        pick(rows, 4),     # медиана ВБП камизестрант + CDK4/6i 16.0 (95% ДИ 12.7–18.2)
        pick(rows, 5),     # медиана ВБП ИА + CDK4/6i 9.2 (95% ДИ 7.2–9.5)
        pick(rows, 6),     # HR 0.44 (95% ДИ 0.31–0.60)
        pick(rows, 7),     # p — как на слайде: <0.00001
    ],
    slide_problems=[
        "p для HR на слайде <0.00001, в NEJM 2025 — <0.0001",
    ],
    questions=[
        "есть ли в исследовании смещение, связанное с особенностями популяции в выборке?",
        "можем ли мы объективно судить об устойчивости этих результатов с учетом того, что в конце наблюдения осталось малопациентов?"
    ],
)
show_gold(path)

In [ ]:
path = save_gold(
    "02_ev302_os",
    study={"name": "EV-302", "nct": "NCT04223856",
           "source": "Powles, ASCO 2026 abstr 4507 (срез 06.10.2025); первичная — Powles NEJM 2024 (на слайде название не напечатано)"},
    key_claims=[
        pick(rows, 4),     # медиана ОВ EV+P 33.6 (95% ДИ 26.6–39.8)
        pick(rows, 5),     # медиана ОВ химиотерапия 15.9 (95% ДИ 13.6–18.3)
        pick(rows, 6),     # HR 0.53 (95% ДИ 0.45–0.63)
        pick(rows, 13),    # 42-мес ОВ EV+P 44.0
        pick(rows, 14),    # 42-мес ОВ химиотерапия 24.6
    ],
    slide_problems=[],
    questions=[
        "ВАШ ВОПРОС 1",
    ],
)
show_gold(path)

In [ ]:
def rows_for(key):
    """Таблица сверки Opus против Sonnet для одного слайда."""
    return compare_drafts(quick_drafts[key]["opus"]["facts"], quick_drafts[key]["sonnet"]["facts"])
    

In [ ]:
rows05 = rows_for("05_bezuclastinib_orr")
show_check(slides["05_bezuclastinib_orr"], rows05, run_ocr(slides["05_bezuclastinib_orr"]), "Opus", "Sonnet")

In [ ]:
path = save_gold(
    "05_bezuclastinib_orr",
    study={"name": "Peak", "nct": None,
           "source": "Wagner, ASCO 2026 abstr 11500 (срез 30.09.2025); точные ЧОО — пресс-релиз Cogent"},
    key_claims=[
        pick(rows05, 2),     # ЧОО безукластиниб + сунитиниб 45.6 (95% ДИ 38.6–52.7)
        pick(rows05, 3),     # ЧОО сунитиниб 25.8 (95% ДИ 20.0–32.3)
        pick(rows05, 4),     # p < 0.0001
        pick(rows05, 27),    # медиана длительности ответа 15.7
        pick(rows05, 28),    # медиана длительности ответа сунитиниб 12.0
    ],
    slide_problems=[
        "ЧОО 45.6/25.8 с ДИ — по пресс-релизу компании; в тезисе ASCO округлено 46/26",
    ],
    questions=[
        "ВАШ ВОПРОС 1",
    ],
)
show_gold(path)